# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [21]:
!cat publications.xlsx

?�9L�ҙ�sbgٮ|�l!��USh9i�b�r:"y_dl��D���|-N��R"4�2�G�%��Z�4�˝y�7	ë��ɂ���  �� PK    ! �)���  I     xl/workbook.xml�Umo�0�>i�!��B���                                                                                                                                                                                                                                                                                                                                                                                                                                  ���N�0E�H�C�-Jܲ5��*Q>�ēƪc[�ii����B�j7���{2��h�nm���ƻR����U^7/���%��rZY�@1__�f� �q��R4D�AJ�h>����V�ƹ�Z�9����NV�8ʩ����ji){^��-I�"{�v^�P!XS)bR�r��K�s(�3�`c�0��������7M4�����ZƐk+�|\|z�(���P��6h_-[�@�!��� Pk���2n�}�?�L��� ��%���d����dN"m,�ǞDO97*�~��ɸ8�O�c|n���E������B��!$}�����;{���[����2�  �� PK    ! �U0#�   L  _rels/.rels �(�                                                           

## Import pandas

We are using the very handy pandas library for dataframes.

In [22]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [23]:
publications = pd.read_excel("publications.xlsx", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url,slides_url
0,1980-10-21,Wonchan Choi,Journal 1,This paper is about the number 1. The number 2...,"Your Name, You. (2009). ""Paper Title Number 1....",paper-title-number-1,http://academicpages.github.io/files/paper1.pdf,http://academicpages.github.io/files/slides1.pdf
1,1982-11-03,Jin A Park,Journal 1,This paper is about the number 2. The number 3...,"Your Name, You. (2010). ""Paper Title Number 2....",paper-title-number-2,http://academicpages.github.io/files/paper2.pdf,http://academicpages.github.io/files/slides2.pdf
2,2017-05-19,Eunjoon Choi,Journal 1,This paper is about the number 3. The number 4...,"Choi, W",paper-title-number-3,http://academicpages.github.io/files/paper3.pdf,http://academicpages.github.io/files/slides3.pdf
3,2020-03-02,Eunsuh Choi,Journal 1,This paper is about the number 3. The number 4...,"Choi, W",paper-title-number-3,http://academicpages.github.io/files/paper3.pdf,http://academicpages.github.io/files/slides3.pdf


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [24]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [35]:
import os
for row, item in publications.iterrows():
    
    date_str = item.pub_date.strftime('%Y-%m-%d')  # Format the Timestamp as 'YYYY-MM-DD'
    md_filename = date_str + "-" + str(item.url_slug) + ".md"
    html_filename = date_str + "-" + str(item.url_slug)
    
    # md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    # html_filename = str(item.pub_date) + "-" + item.url_slug
    # year = item.pub_date.year
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(date_str) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.slides_url)) > 5:
        md += "\nslidesurl: '" + item.slides_url + "'"

    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"

    if len(str(item.slides_url)) > 5:
        md += "\n[Download slides here](" + item.slides_url + ")\n" 

    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [29]:
!ls ../_publications/

1980-10-21-paper-title-number-1.md 2017-05-19-paper-title-number-3.md
1982-11-03-paper-title-number-2.md 2020-03-02-paper-title-number-3.md


In [33]:
!cat ../_publications/1980-10-21-paper-title-number-1.md 2017-05-19-paper-title-number-3.md

---
title: "Wonchan Choi"
collection: publications
permalink: /publication/1980-10-21-paper-title-number-1
excerpt: 'This paper is about the number 1. The number 2 is left for future work.'
date: 1980-10-21 00:00:00
venue: 'Journal 1'
slidesurl: 'http://academicpages.github.io/files/slides1.pdf'
paperurl: 'http://academicpages.github.io/files/paper1.pdf'
citation: 'Your Name, You. (2009). &quot;Paper Title Number 1.&quot; <i>Journal 1</i>. 1(1).'
---
This paper is about the number 1. The number 2 is left for future work.

[Download slides here](http://academicpages.github.io/files/slides1.pdf)

[Download paper here](http://academicpages.github.io/files/paper1.pdf)

Recommended citation: Your Name, You. (2009). "Paper Title Number 1." <i>Journal 1</i>. 1(1).cat: 2017-05-19-paper-title-number-3.md: No such file or directory
